# TEMP — ZIT-ET HPO v3 (Colab, refit + 산출물 포함)

ZITETRegressor의 **Optuna HPO + 5-fold refit + 후처리 + 산출물 저장**까지 풀 파이프라인.

- **차이 vs v2**: anchor를 v2 best HP(trial #41, OOF=0.005509) 기반으로 갱신 + v2에서 발견된 boundary 4개 search range 확장
- **anchor 변경 요약 (v2 → v3, v2 best #41 적용)**:
  - `zeta` 1.79→**1.563**, `mu_n_estimators` 53→**154** (μ는 무거워짐), `mu_max_depth` 9→12
  - `pi_n_estimators` 137→**72** (π는 가벼움), `pi_max_depth` 12→**20** (상한), `pi_max_features` 0.149→**0.278** (상한), `pi_bootstrap` True→False
  - `phi_max_depth` 9→**5** (하한), `phi_max_features` 0.130→**0.038** (하한)
  - `tau_pi` 0.80→**0.889**
- **Search range 확장 (v2 boundary 4개 해결)**:
  - `pi_max_depth`: [5, 20] → [5, **30**]
  - `pi_max_features`: [0.03, 0.30] → [0.03, **0.50**]
  - `phi_max_depth`: [5, 12] → [**3**, 12] (φ가 더 얕게 가고 싶어함)
  - `phi_max_features`: [0.03, 0.15] → [**0.01**, 0.15] (φ가 더 적은 feature)
- **나머지 range 유지** (v2 boundary 약했던 `mu_min_samples_split` 등은 그대로)
- **출력**: `4_output/01_zit/temp_zit_et_hpo_v3/` — best_params.json, fold_models.pkl, oof/val/test ×{die,unit}.csv, optuna_*.db → Colab zip 다운로드

## 1. 환경 설정 + import

In [1]:
import os, sys

GDRIVE_CODE_ID         = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'
GDRIVE_DATASET_ID      = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'
GDRIVE_MODELING_ID     = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # zit.py 포함된 modeling.zip (이미 업로드됨)

try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/zit.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../setup.py

# ── 안전망: %run 실패 시에도 sys.path 보장 ──
def _ensure_sys_path():
    cur = os.getcwd()
    for _ in range(8):
        if os.path.exists(os.path.join(cur, 'setup.py')) and os.path.isdir(os.path.join(cur, 'utils')):
            for sub in ['', '2_preprocessing', '3_modeling']:
                p = os.path.join(cur, sub) if sub else cur
                if p not in sys.path and os.path.isdir(p):
                    sys.path.insert(0, p)
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    if os.path.exists('/content/project/setup.py'):
        for sub in ['', '2_preprocessing', '3_modeling']:
            p = os.path.join('/content/project', sub) if sub else '/content/project'
            if p not in sys.path and os.path.isdir(p):
                sys.path.insert(0, p)
        return '/content/project'
    return None
_proj_root = _ensure_sys_path()
print(f'[sys.path] project_root={_proj_root}')

# ── 표준 import ──
import time, json
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.ensemble import ExtraTreesRegressor
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# ── 프로젝트 utils ──
from utils.config import SEED, OUTPUT_DIR, TARGET_COL, KEY_COL, DIE_KEY_COL
from utils.data import load_all, get_feat_cols, split_xs

# ── 프로젝트 modules ──
from modules import preprocess, hpo, postprocess
from meta_features import add_meta_features   # 2_preprocessing/meta_features.py
from modules.zit import ZITboostRegressor      # 인라인 subclass의 부모

print(f'SEED={SEED}, OUTPUT_DIR={OUTPUT_DIR}')

패키지 설치 중: ['catboost', 'optuna', 'boruta', 'pytorch-tabnet', 'rtdl-revisiting-models']
setup 완료
[sys.path] project_root=/content/project
SEED=42, OUTPUT_DIR=/content/project/4_output


## 2. 실험 설정 (경량 anchor + 좁힌 search space)

ET는 LGBM과 달리 트리가 끝까지 자라므로 `n_estimators × max_depth × max_features` 곱이 그대로 시간으로 옴. anchor를 LGBM 1차 anchor 수준으로 잡으면 1 fold = 수십 분 걸림. **경량 anchor + EM iter도 줄임 (10→3)**으로 1 trial ~30~60분 목표.

In [2]:
EXP_ID = 'temp-zit-et-hpo-003'   # ★ 003 — v2 best HP 기반 anchor + boundary 4개 range 확장
USER   = 'jh'

# ── HPO 예산 ──
N_TRIALS         = 100
N_FOLDS          = 5
N_STARTUP_TRIALS = 20
N_JOBS           = -1
N_JOBS_OPTUNA    = 1
TIMEOUT_SEC      = 14 * 3600

OUT_DIR = os.path.join(OUTPUT_DIR, '01_zit', 'temp_zit_et_hpo_v3')   # ★ v3 폴더
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')

CLIP_Y_EXTREME = True

# ── PP_FIXED ──
PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

# ── ZIT-ET anchor (v2 best trial #41 그대로, OOF=0.005509) ──
ZIT_ET_ANCHOR = {
    'zeta':                  1.563,
    'n_em_iters':            3,
    # μ (v2 best: 더 무거워짐 — n_estim 53→154, depth 9→12)
    'mu_n_estimators':       154,
    'mu_criterion':          'squared_error',
    'mu_max_depth':          12,
    'mu_min_samples_split':  11,
    'mu_min_samples_leaf':   23,
    'mu_max_features':       0.0648,
    'mu_bootstrap':          False,
    # π (v2 best: 가벼워짐 — n_estim 137→72, but depth 12→20 상한, max_feat 0.149→0.278 상한)
    'pi_n_estimators':       72,
    'pi_max_depth':          20,
    'pi_min_samples_split':  42,
    'pi_min_samples_leaf':   28,
    'pi_max_features':       0.278,
    'pi_bootstrap':          False,
    # φ (v2 best: 더 얕게 — depth 9→5 하한, max_feat 0.130→0.038 하한)
    'phi_n_estimators':      115,
    'phi_max_depth':         5,
    'phi_min_samples_split': 24,
    'phi_min_samples_leaf':  12,
    'phi_max_features':      0.0381,
    'phi_bootstrap':         True,
}
ANCHOR_TAU_PI = 0.889
TAU_PI_RANGE  = (0.70, 0.95)

# ── Search space (v2 boundary 4개 해결: π 더 깊고 넓게, φ 더 얕고 좁게) ──
def sample_zit_et_space(trial):
    return {
        'zeta':                   trial.suggest_float('zeta', 1.2, 1.95),
        'n_em_iters':             trial.suggest_int('n_em_iters', 2, 5),
        # μ — 변경 없음
        'mu_n_estimators':        trial.suggest_int('mu_n_estimators', 30, 200),
        'mu_criterion':           trial.suggest_categorical('mu_criterion', ['squared_error', 'poisson']),
        'mu_max_depth':           trial.suggest_int('mu_max_depth', 6, 14),
        'mu_min_samples_split':   trial.suggest_int('mu_min_samples_split', 10, 50),
        'mu_min_samples_leaf':    trial.suggest_int('mu_min_samples_leaf', 10, 50),
        'mu_max_features':        trial.suggest_float('mu_max_features', 0.03, 0.15),
        'mu_bootstrap':           trial.suggest_categorical('mu_bootstrap', [True, False]),
        # π — 상한 크게 확장
        'pi_n_estimators':        trial.suggest_int('pi_n_estimators', 50, 250),
        'pi_max_depth':           trial.suggest_int('pi_max_depth', 5, 30),         # ↑ 20 → 30 (v2 best=20 상한)
        'pi_min_samples_split':   trial.suggest_int('pi_min_samples_split', 10, 50),
        'pi_min_samples_leaf':    trial.suggest_int('pi_min_samples_leaf', 10, 50),
        'pi_max_features':        trial.suggest_float('pi_max_features', 0.03, 0.50),   # ↑ 0.30 → 0.50 (v2 best=0.278 상한)
        'pi_bootstrap':           trial.suggest_categorical('pi_bootstrap', [True, False]),
        # φ — 하한 더 낮춤
        'phi_n_estimators':       trial.suggest_int('phi_n_estimators', 30, 150),
        'phi_max_depth':          trial.suggest_int('phi_max_depth', 3, 12),        # ↓ 5 → 3 (v2 best=5 하한)
        'phi_min_samples_split':  trial.suggest_int('phi_min_samples_split', 10, 50),
        'phi_min_samples_leaf':   trial.suggest_int('phi_min_samples_leaf', 10, 50),
        'phi_max_features':       trial.suggest_float('phi_max_features', 0.01, 0.15),  # ↓ 0.03 → 0.01 (v2 best=0.038 하한)
        'phi_bootstrap':          trial.suggest_categorical('phi_bootstrap', [True, False]),
    }

print(f'EXP_ID: {EXP_ID}')
print(f'OUT_DIR: {OUT_DIR}')
print(f'DB_PATH: {DB_PATH}')
print(f'N_TRIALS: {N_TRIALS}, N_FOLDS: {N_FOLDS}, TIMEOUT_SEC: {TIMEOUT_SEC}s ({TIMEOUT_SEC/3600:.1f}h)')

EXP_ID: temp-zit-et-hpo-003
OUT_DIR: /content/project/4_output/01_zit/temp_zit_et_hpo_v3
DB_PATH: /content/project/4_output/01_zit/temp_zit_et_hpo_v3/optuna_jh_temp-zit-et-hpo-003.db
N_TRIALS: 100, N_FOLDS: 5, TIMEOUT_SEC: 50400s (14.0h)


## 3. 인라인 ZITETRegressor 정의

`modules.zit.ZITboostRegressor` 상속, `_m_step`만 LGBM → ExtraTreesRegressor로 교체. (`zit_et.ipynb`와 동일)

In [3]:
class ZITETRegressor(ZITboostRegressor):
    """ZITboost의 LGBM 3종을 ExtraTreesRegressor로 교체한 변형.

    부모(modules.zit.ZITboostRegressor)의 EM 흐름·Tweedie 가정·predict는 그대로.
    `_m_step`만 ExtraTreesRegressor로 override.
    """
    def __init__(
        self,
        zeta=1.5, n_em_iters=3, em_tol=1e-7,
        # μ
        mu_n_estimators=100, mu_criterion='squared_error', mu_max_depth=None,
        mu_min_samples_split=20, mu_min_samples_leaf=30, mu_max_features=0.05, mu_bootstrap=False,
        # π
        pi_n_estimators=80, pi_max_depth=None,
        pi_min_samples_split=30, pi_min_samples_leaf=30, pi_max_features=0.05, pi_bootstrap=False,
        # φ
        phi_n_estimators=80, phi_max_depth=None,
        phi_min_samples_split=30, phi_min_samples_leaf=30, phi_max_features=0.05, phi_bootstrap=False,
        random_state=SEED, n_jobs=-1, verbose=-1,
    ):
        # ★ 부모 __init__ 호출 안 함 (LGBM-specific attr 회피)
        self.zeta = zeta
        self.n_em_iters = n_em_iters
        self.em_tol = em_tol
        # μ
        self.mu_n_estimators = mu_n_estimators
        self.mu_criterion = mu_criterion
        self.mu_max_depth = mu_max_depth
        self.mu_min_samples_split = mu_min_samples_split
        self.mu_min_samples_leaf = mu_min_samples_leaf
        self.mu_max_features = mu_max_features
        self.mu_bootstrap = mu_bootstrap
        # π
        self.pi_n_estimators = pi_n_estimators
        self.pi_max_depth = pi_max_depth
        self.pi_min_samples_split = pi_min_samples_split
        self.pi_min_samples_leaf = pi_min_samples_leaf
        self.pi_max_features = pi_max_features
        self.pi_bootstrap = pi_bootstrap
        # φ
        self.phi_n_estimators = phi_n_estimators
        self.phi_max_depth = phi_max_depth
        self.phi_min_samples_split = phi_min_samples_split
        self.phi_min_samples_leaf = phi_min_samples_leaf
        self.phi_max_features = phi_max_features
        self.phi_bootstrap = phi_bootstrap
        # 공통
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.verbose = verbose

    def _mu_params(self):
        return dict(
            n_estimators=self.mu_n_estimators, criterion=self.mu_criterion,
            max_depth=self.mu_max_depth,
            min_samples_split=self.mu_min_samples_split,
            min_samples_leaf=self.mu_min_samples_leaf,
            max_features=self.mu_max_features, bootstrap=self.mu_bootstrap,
            random_state=self.random_state, n_jobs=self.n_jobs,
        )

    def _pi_params(self):
        return dict(
            n_estimators=self.pi_n_estimators, criterion='squared_error',
            max_depth=self.pi_max_depth,
            min_samples_split=self.pi_min_samples_split,
            min_samples_leaf=self.pi_min_samples_leaf,
            max_features=self.pi_max_features, bootstrap=self.pi_bootstrap,
            random_state=self.random_state, n_jobs=self.n_jobs,
        )

    def _phi_params(self):
        return dict(
            n_estimators=self.phi_n_estimators, criterion='squared_error',
            max_depth=self.phi_max_depth,
            min_samples_split=self.phi_min_samples_split,
            min_samples_leaf=self.phi_min_samples_leaf,
            max_features=self.phi_max_features, bootstrap=self.phi_bootstrap,
            random_state=self.random_state, n_jobs=self.n_jobs,
        )

    def _m_step(self, X, y, posterior):
        """posterior Π 고정 → 3개 ExtraTreesRegressor 순서대로 재학습."""
        w_tw = 1 - posterior

        et_pi = ExtraTreesRegressor(**self._pi_params())
        et_pi.fit(X, posterior)
        pi_pred = np.clip(et_pi.predict(X), 1e-8, 1 - 1e-8)

        phi_for_weight = np.maximum(self._phi_current, 1e-10)
        mu_weight = w_tw / phi_for_weight

        et_mu = ExtraTreesRegressor(**self._mu_params())
        et_mu.fit(X, y, sample_weight=mu_weight)
        mu_pred = np.maximum(et_mu.predict(X), 1e-10)

        residual_sq = np.square(y - mu_pred)
        mu_pow_zeta = np.power(mu_pred, self.zeta)
        phi_target = np.clip(residual_sq / np.maximum(mu_pow_zeta, 1e-10), 1e-8, 1e6)

        et_phi = ExtraTreesRegressor(**self._phi_params())
        et_phi.fit(X, phi_target, sample_weight=w_tw)
        phi_pred = np.clip(et_phi.predict(X), 1e-8, 1e6)

        # 부모의 fit/predict_components가 lgb_* 이름으로 저장·참조하므로 같은 이름으로 반환
        return et_pi, et_mu, et_phi, pi_pred, mu_pred, phi_pred


_test = ZITETRegressor(n_em_iters=2, mu_n_estimators=10, pi_n_estimators=10, phi_n_estimators=10)
print(f'[OK] ZITETRegressor 정의 완료 — params: {len(_test.get_params())}')
del _test

[OK] ZITETRegressor 정의 완료 — params: 25


## 4. 데이터 로드 + Y clip + PP_FIXED 적용 (1회)

In [4]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_clean,
    position_mode='raw', use_die_xy=True,
)

# train/val/test 모두 numpy 변환 (refit에 필요)
X_train = xs_train[feat_cols_clean].values.astype(np.float64)
X_val   = xs_val[feat_cols_clean].values.astype(np.float64)
X_test  = xs_test[feat_cols_clean].values.astype(np.float64)

uid_train_die = xs_train[KEY_COL].values
uid_val_die   = xs_val[KEY_COL].values
uid_test_die  = xs_test[KEY_COL].values

y_train_unit_s = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit_s   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit_s  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

y_train_die = xs_train[KEY_COL].map(y_train_unit_s).values.astype(np.float64)

print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')
print(f'  X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')
print(f'  unit train={len(y_train_unit_s):,}, val={len(y_val_unit_s):,}, test={len(y_test_unit_s):,}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개 샘플
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1031 (56개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1031
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 926개
    컬럼: 1031 → 926 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=30%
  제거: 5개, 잔여: 921개
    컬럼: 926 → 921 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 894개
    컬럼: 921 → 894 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 330개, 잔여: 564개
    컬럼: 894 → 564 (330개 제거)
    DataFrame: (104748, 624)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행


/content/project/2_preprocessing/cleaning.py:373: RuntimeWarning: invalid value encountered in divide
  avg = weighted_sum / w_sum             # (C,)


  1단계 (공간 보간, dist<=6.0): 161,870개 채움 → 잔여: 181,624
  2단계 (lot 평균, train 기준): 100,428개 채움 → 잔여: 81,196
  3단계 (train 전체 평균): 81,196개 채움 → 잔여: 0

  [요약] 343,494 → 공간(161,870) → lot(100,428) → 전체(81,196) → 잔여(0)

[고상관 제거] threshold=0.96, keep_by=std (std)
  제거: 0개, 잔여: 564개
    [고상관 제거 2차 / imputation 후] threshold=0.96
    컬럼: 564 → 564 (0개 제거)
    DataFrame: (104748, 633)

클리닝 완료: 1031 → 564 features (467개 제거)
  + indicator 컬럼: 9개 → 총 573개
  train: (104748, 633)
  val:   (34908, 633)
  test:  (34916, 633)
이상치 처리 파이프라인 시작 (method=winsorize)
[이상치 탐지] IQR × 1.5
  이상치 > 5%: 112개
  이상치 > 10%: 64개
[Winsorization] lower=0%, upper=99%
  적용 feature: 573개

이상치 처리 완료 (method=winsorize)
  train: (104748, 633)


/content/project/2_preprocessing/meta_features.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  xs[lot_c] = split[0]
/content/project/2_preprocessing/meta_features.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  xs[wf_c] = split[1]
/content/project/2_preprocessing/meta_features.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented

[add_meta_features] position_mode='raw', use_die_xy=True, use_loc_x_ohe=False → position=['position'], die_xy=['die_x', 'die_y'] (feat_cols: 576)

[전처리 완료] feat_cols: 576
  X_train: (104748, 576), X_val: (34908, 576), X_test: (34916, 576)
  unit train=26,187, val=8,727, test=8,729


## 5. K-fold + Optuna objective

- KFold는 unit ID 단위 분할 (leakage 방지)
- 매 trial: 5 fold × ZITETRegressor 학습 → die π/μ → τ_π → unit 평균 → OOF unit RMSE

In [5]:
unique_units = y_train_unit_s.index.values
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf.split(unique_units))


def _mean_die_to_unit(pred_die, uid_die):
    df = pd.DataFrame({KEY_COL: uid_die, 'pred': pred_die})
    return df.groupby(KEY_COL, sort=False)['pred'].mean().reset_index()

def _apply_tau_pi(pred_die, pi_die, tau_pi):
    return np.where(pi_die > tau_pi, 0.0, pred_die)


def objective(trial):
    t0 = time.time()

    params = sample_zit_et_space(trial)
    tau_pi = trial.suggest_float('tau_pi', TAU_PI_RANGE[0], TAU_PI_RANGE[1])

    params['random_state'] = SEED
    params['n_jobs']       = N_JOBS
    params['verbose']      = -1
    params['em_tol']       = 1e-7

    fold_oof_rmse = []
    oof_pred_unit = pd.Series(np.nan, index=y_train_unit_s.index, dtype=np.float64)

    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        tr_mask = np.isin(uid_train_die, tr_units)
        vl_mask = np.isin(uid_train_die, vl_units)

        model = ZITETRegressor(**params)
        model.fit(X_train[tr_mask], y_train_die[tr_mask])

        pi_vl, mu_vl, _ = model.predict_components(X_train[vl_mask])
        pred_die_raw = np.clip((1 - pi_vl) * mu_vl, 0, None)
        pred_die_taupi = _apply_tau_pi(pred_die_raw, pi_vl, tau_pi)
        unit_pred_df = _mean_die_to_unit(pred_die_taupi, uid_train_die[vl_mask])

        oof_pred_unit.loc[unit_pred_df[KEY_COL].values] = unit_pred_df['pred'].values
        y_vl = y_train_unit_s.loc[unit_pred_df[KEY_COL].values].values
        fold_rmse = float(np.sqrt(np.mean((unit_pred_df['pred'].values - y_vl) ** 2)))
        fold_oof_rmse.append(fold_rmse)

        avg = float(np.mean(fold_oof_rmse))
        trial.report(avg, step=fold_idx)
        if trial.should_prune():
            trial.set_user_attr('pruned_at_fold', fold_idx + 1)
            trial.set_user_attr('elapsed_sec', time.time() - t0)
            trial.set_user_attr('tau_pi', tau_pi)
            raise optuna.TrialPruned()

    if oof_pred_unit.isna().any():
        raise RuntimeError('OOF NaN — fold 누락')

    oof_rmse = float(np.sqrt(np.mean((oof_pred_unit.values - y_train_unit_s.values) ** 2)))
    elapsed = time.time() - t0
    trial.set_user_attr('elapsed_sec', elapsed)
    trial.set_user_attr('tau_pi', tau_pi)
    trial.set_user_attr('fold_oof_rmse', fold_oof_rmse)
    print(f'  trial #{trial.number}: τ_π={tau_pi:.3f}, oof={oof_rmse:.6f}, elapsed={elapsed:.0f}s')
    return oof_rmse


print(f'fold split: {N_FOLDS} folds, unit 단위 분할, seed={SEED}')

fold split: 5 folds, unit 단위 분할, seed=42


## 6. Optuna study + anchor enqueue + optimize

In [6]:
sampler = TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS)
pruner  = MedianPruner(n_startup_trials=N_STARTUP_TRIALS, n_warmup_steps=2)

study = optuna.create_study(
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    sampler=sampler, pruner=pruner,
    direction='minimize', load_if_exists=True,   # 세션 끊겨도 같은 db로 재시작 가능
)

ANCHOR_FOR_ENQUEUE = dict(ZIT_ET_ANCHOR)
ANCHOR_FOR_ENQUEUE['tau_pi'] = ANCHOR_TAU_PI
if len(study.trials) == 0:
    study.enqueue_trial(ANCHOR_FOR_ENQUEUE)
    print(f'[enqueue] ZIT_ET_ANCHOR + tau_pi={ANCHOR_TAU_PI} 첫 trial 강제')
else:
    print(f'[enqueue skip] 기존 trial {len(study.trials)} 있음 — resume')

study_meta = {
    'exp_id': EXP_ID, 'user': USER, 'model': 'ZITETRegressor (zit_only, hpo only)',
    'n_trials': N_TRIALS, 'n_folds': N_FOLDS, 'n_jobs': N_JOBS,
    'pp_fixed': PP_FIXED, 'anchor': ZIT_ET_ANCHOR, 'anchor_tau_pi': ANCHOR_TAU_PI,
    'sampler': 'TPE seed=None multivariate group',
    'pruner':  f'MedianPruner n_startup={N_STARTUP_TRIALS} n_warmup=2',
    'CLIP_Y_EXTREME': CLIP_Y_EXTREME, 'SEED': int(SEED),
}
for k, v in study_meta.items():
    study.set_user_attr(k, str(v))

print(f'study: {study.study_name}, DB: {DB_PATH}')
print(f'기존 trial: {len(study.trials)}')

t_start = time.time()
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, n_jobs=N_JOBS_OPTUNA, show_progress_bar=True)
print(f'\n[HPO 완료] 전체 {time.time()-t_start:.0f}s, total trials={len(study.trials)}')
if len(study.trials) > 0:
    print(f'  best OOF RMSE: {study.best_value:.6f}')

/tmp/ipykernel_11224/2583575105.py:1: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS)
/tmp/ipykernel_11224/2583575105.py:1: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  sampler = TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS)
[I 2026-05-17 05:33:05,841] A new study created in RDB with name: temp-zit-et-hpo-003


[enqueue] ZIT_ET_ANCHOR + tau_pi=0.889 첫 trial 강제
study: temp-zit-et-hpo-003, DB: /content/project/4_output/01_zit/temp_zit_et_hpo_v3/optuna_jh_temp-zit-et-hpo-003.db
기존 trial: 1


  0%|          | 0/100 [00:00<?, ?it/s]

  trial #0: τ_π=0.889, oof=0.005511, elapsed=1399s
[I 2026-05-17 05:56:25,086] Trial 0 finished with value: 0.005510857697683313 and parameters: {'zeta': 1.563, 'n_em_iters': 3, 'mu_n_estimators': 154, 'mu_criterion': 'squared_error', 'mu_max_depth': 12, 'mu_min_samples_split': 11, 'mu_min_samples_leaf': 23, 'mu_max_features': 0.0648, 'mu_bootstrap': False, 'pi_n_estimators': 72, 'pi_max_depth': 20, 'pi_min_samples_split': 42, 'pi_min_samples_leaf': 28, 'pi_max_features': 0.278, 'pi_bootstrap': False, 'phi_n_estimators': 115, 'phi_max_depth': 5, 'phi_min_samples_split': 24, 'phi_min_samples_leaf': 12, 'phi_max_features': 0.0381, 'phi_bootstrap': True, 'tau_pi': 0.889}. Best is trial 0 with value: 0.005510857697683313.
  trial #1: τ_π=0.706, oof=0.005645, elapsed=1104s
[I 2026-05-17 06:14:49,685] Trial 1 finished with value: 0.005645027484634906 and parameters: {'zeta': 1.9157549630364203, 'n_em_iters': 4, 'mu_n_estimators': 100, 'mu_criterion': 'poisson', 'mu_max_depth': 8, 'mu_min_sam

## 7. Best trial 정보 + anchor 검증

(DB 다운로드는 마지막 셀에서 산출물 zip과 함께)

In [7]:
best_trial = study.best_trial
best_params = best_trial.params
best_tau_pi = float(best_trial.user_attrs.get('tau_pi', ANCHOR_TAU_PI))

print(f'=== Best Trial #{best_trial.number} ===')
print(f'  OOF RMSE  : {best_trial.value:.6f}')
print(f'  best τ_π  : {best_tau_pi:.4f}')
print(f'  elapsed   : {best_trial.user_attrs.get("elapsed_sec", 0):.0f}s')
for k, v in sorted(best_params.items()):
    print(f'    {k}: {v}')

# anchor 검증 (trial 0 == anchor?)
trial0 = study.trials[0]
anchor_check = all(
    abs(float(trial0.params.get(k, np.nan)) - float(v)) < 1e-9
    if not isinstance(v, (str, bool)) else trial0.params.get(k) == v
    for k, v in ANCHOR_FOR_ENQUEUE.items()
)
print(f'\n[검증] trial 0 == anchor? {anchor_check}')

print(f'\n[trial 통계]')
n_complete = sum(1 for t in study.trials if t.state.name == 'COMPLETE')
n_pruned   = sum(1 for t in study.trials if t.state.name == 'PRUNED')
n_failed   = sum(1 for t in study.trials if t.state.name == 'FAIL')
print(f'  COMPLETE: {n_complete}, PRUNED: {n_pruned}, FAIL: {n_failed}')

=== Best Trial #27 ===
  OOF RMSE  : 0.005505
  best τ_π  : 0.9169
  elapsed   : 1960s
    mu_bootstrap: False
    mu_criterion: poisson
    mu_max_depth: 8
    mu_max_features: 0.06943263319935303
    mu_min_samples_leaf: 21
    mu_min_samples_split: 20
    mu_n_estimators: 192
    n_em_iters: 2
    phi_bootstrap: True
    phi_max_depth: 8
    phi_max_features: 0.04458225854989846
    phi_min_samples_leaf: 19
    phi_min_samples_split: 23
    phi_n_estimators: 101
    pi_bootstrap: False
    pi_max_depth: 22
    pi_max_features: 0.4700411688471291
    pi_min_samples_leaf: 29
    pi_min_samples_split: 41
    pi_n_estimators: 172
    tau_pi: 0.9169337967036683
    zeta: 1.6671049737824424

[검증] trial 0 == anchor? True

[trial 통계]
  COMPLETE: 33, PRUNED: 0, FAIL: 0


## 8. Best HP 5-fold refit + die-level π/μ 캡처

best HP로 5 fold를 다시 학습 → train OOF + val/test die-level π/μ/pred 확보.

In [8]:
# best HP에서 τ_π를 빼고(모델 인자가 아님) 모델 고정 인자를 다시 보강
best_full_params = {k: v for k, v in best_params.items() if k != 'tau_pi'}
best_full_params['random_state'] = SEED
best_full_params['n_jobs']       = N_JOBS
best_full_params['verbose']      = -1
best_full_params['em_tol']       = 1e-7
# ★ ET는 device 인자 없음 (LGBM과 차이)

n_train_die = len(X_train)
n_val_die   = len(X_val)
n_test_die  = len(X_test)

oof_die_pi   = np.full(n_train_die, np.nan)
oof_die_mu   = np.full(n_train_die, np.nan)
oof_die_pred = np.full(n_train_die, np.nan)

val_die_pi   = np.zeros(n_val_die)
val_die_mu   = np.zeros(n_val_die)
val_die_pred = np.zeros(n_val_die)
test_die_pi   = np.zeros(n_test_die)
test_die_mu   = np.zeros(n_test_die)
test_die_pred = np.zeros(n_test_die)

fold_models = []
em_history_per_fold = []

print(f'=== Best HP 5-fold refit ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    tr_units = unique_units[tr_uidx]
    vl_units = unique_units[vl_uidx]
    tr_mask = np.isin(uid_train_die, tr_units)
    vl_mask = np.isin(uid_train_die, vl_units)

    model = ZITETRegressor(**best_full_params)
    model.fit(X_train[tr_mask], y_train_die[tr_mask])

    pi_vl, mu_vl, _ = model.predict_components(X_train[vl_mask])
    oof_die_pi[vl_mask]   = pi_vl
    oof_die_mu[vl_mask]   = mu_vl
    oof_die_pred[vl_mask] = np.clip((1 - pi_vl) * mu_vl, 0, None)

    pi_v, mu_v, _ = model.predict_components(X_val)
    pi_t, mu_t, _ = model.predict_components(X_test)
    val_die_pi    += pi_v / N_FOLDS
    val_die_mu    += mu_v / N_FOLDS
    val_die_pred  += np.clip((1 - pi_v) * mu_v, 0, None) / N_FOLDS
    test_die_pi   += pi_t / N_FOLDS
    test_die_mu   += mu_t / N_FOLDS
    test_die_pred += np.clip((1 - pi_t) * mu_t, 0, None) / N_FOLDS

    fold_models.append(model)
    em_history_per_fold.append(model.em_history_)
    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s)')

assert not np.isnan(oof_die_pred).any(), 'OOF die pred 미커버'

print(f'\n[EM 수렴 체크]')
for f, hist in enumerate(em_history_per_fold):
    rmses = [h['rmse'] for h in hist]
    monotonic = all(rmses[i+1] <= rmses[i] + 1e-6 for i in range(len(rmses)-1))
    print(f'  fold {f+1}: {len(hist)} EM iter, last_rmse={rmses[-1]:.6f}, monotonic_decreasing={monotonic}')

print(f'\n[refit 완료] die-level π/μ/pred 캡처 OK')

=== Best HP 5-fold refit ===
  fold 1/5 done (403s)
  fold 2/5 done (799s)
  fold 3/5 done (1195s)
  fold 4/5 done (1588s)
  fold 5/5 done (1985s)

[EM 수렴 체크]
  fold 1: 2 EM iter, last_rmse=0.005031, monotonic_decreasing=False
  fold 2: 2 EM iter, last_rmse=0.005150, monotonic_decreasing=False
  fold 3: 2 EM iter, last_rmse=0.005116, monotonic_decreasing=False
  fold 4: 2 EM iter, last_rmse=0.005156, monotonic_decreasing=False
  fold 5: 2 EM iter, last_rmse=0.005105, monotonic_decreasing=False

[refit 완료] die-level π/μ/pred 캡처 OK


## 9. 후처리 — τ_π 적용 → 집계 8 + position Optuna + zero_clip(log)

In [9]:
oof_die_pred_taupi  = _apply_tau_pi(oof_die_pred,  oof_die_pi,  best_tau_pi)
val_die_pred_taupi  = _apply_tau_pi(val_die_pred,  val_die_pi,  best_tau_pi)
test_die_pred_taupi = _apply_tau_pi(test_die_pred, test_die_pi, best_tau_pi)

killed = {
    'oof':  float((oof_die_pi  > best_tau_pi).mean()),
    'val':  float((val_die_pi  > best_tau_pi).mean()),
    'test': float((test_die_pi > best_tau_pi).mean()),
}
print(f'[τ_π={best_tau_pi:.3f} 적용] 0 처리 die 비율: {killed}')

pp_res = postprocess.tune_and_apply(
    xs_train, xs_val, xs_test,
    die_pred_train=oof_die_pred_taupi,
    die_pred_val=val_die_pred_taupi,
    die_pred_test=test_die_pred_taupi,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    use_pi_threshold=False,
    agg_methods=postprocess.AGG_METHODS,
    zero_clip_log_space=True,
    position_method='optuna',
    position_optuna_n_trials=50,
)

print(f'\n[Postprocess]')
print(f'  best_agg            : {pp_res["best_agg"]}')
print(f'  pos_weights         : {pp_res["pos_weights"]}')
print(f'  best_zero_clip(log) : {pp_res["best_zero_clip"]}')
print(f'  position_method     : {pp_res["position_method"]}')
print(f'  train_rmse          : {pp_res["train_rmse"]:.6f}')
print(f'  val_rmse_final      : {pp_res.get("val_rmse_final")}')

if pp_res.get('final_val_unit') is not None:
    _val_pred = pp_res['final_val_unit'].set_index(KEY_COL)['pred'].loc[y_val_unit_s.index]
    val_rmse  = float(np.sqrt(np.mean((_val_pred.values  - y_val_unit_s.values)  ** 2)))
    print(f'  val_rmse            : {val_rmse:.6f}')
if pp_res.get('final_test_unit') is not None:
    _test_pred = pp_res['final_test_unit'].set_index(KEY_COL)['pred'].loc[y_test_unit_s.index]
    test_rmse  = float(np.sqrt(np.mean((_test_pred.values - y_test_unit_s.values) ** 2)))
    print(f'  test_rmse           : {test_rmse:.6f}')

print(f'  agg_rmses           : {pp_res["agg_rmses"]}')
print(f'  decisions           :')
for k, v in pp_res.get('decisions', {}).items():
    print(f'    {k:14s} {v}')

[τ_π=0.917 적용] 0 처리 die 비율: {'oof': 0.07615419864818421, 'val': 0.0718459951873496, 'test': 0.0750085920494902}


/content/project/3_modeling/modules/postprocess.py:178: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(seed=seed, multivariate=True, group=True)
/content/project/3_modeling/modules/postprocess.py:178: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(seed=seed, multivariate=True, group=True)
[I 2026-05-17 20:35:04,518] A new study created in memory with name: no-name-97c39ebc-05f9-4fd7-8417-2791d836e2dc


[Position weights / Optuna 50t] best=0.005505, w=[0.026, 0.333, 0.378, 0.262]
[Aggregation] RMSEs: {'mean': 0.005505, 'median': 0.005505, 'max': 0.005508, 'min': 0.005514, 'trimmed_mean': 0.005505, 'weighted': 0.005505, 'Q25': 0.005508, 'Q75': 0.005505}
[Aggregation] best=weighted (0.005505)
[zero_clip (log)] best=0.0010 (0.005506)
[Postprocess] best_agg=mean, pi_th=None, zero_clip (log)=None, train_rmse=0.005505, val_rmse=0.005721
  baseline_mean                  val_rmse=0.005721072881057739
  after_agg(mean)                val_rmse=0.005721072881057739
  after_pi_th                    val_rmse=0.005721072881057739
  after_zero_clip                val_rmse=0.005721072881057739
  [decision] aggregation    weighted rejected (val 0.005721 <= 0.005722) -> keep mean
  [decision] zero_clip      0.0010 rejected (val 0.005721 <= 0.005721) — skip

[Postprocess]
  best_agg            : mean
  pos_weights         : None
  best_zero_clip(log) : None
  position_method     : optuna
  train_rmse   

## 10. 산출물 저장 + Colab download

저장: `best_params.json`, `fold_models.pkl`, `oof/val/test_die.csv`, `oof/val/test_unit.csv`, `optuna_*.db` — 총 9개. Colab이면 zip으로 묶어 자동 다운로드.

In [10]:
import json, pickle, hashlib

# 1) fold_models.pkl
with open(os.path.join(OUT_DIR, 'fold_models.pkl'), 'wb') as f:
    pickle.dump({
        'fold_models':         fold_models,
        'feature_names':       feat_cols_clean,
        'model_name':          'zit_et',
        'n_folds':             N_FOLDS,
        'em_history_per_fold': em_history_per_fold,
    }, f)

# 2) best_params.json
uid_arr = ys_input['train'][KEY_COL].unique()
unit_ids_hash = hashlib.sha1(','.join(map(str, uid_arr)).encode()).hexdigest()

best_meta = {
    'exp_id':                EXP_ID,
    'model_name':            'zit_et',
    'best_trial_number':     best_trial.number,
    'best_oof_rmse':         float(best_trial.value),
    'best_params_resolved':  best_full_params,
    'best_tau_pi':           best_tau_pi,
    'feature_names':         feat_cols_clean,
    'n_features':            len(feat_cols_clean),
    'n_folds':               N_FOLDS,
    'unit_ids_hash':         unit_ids_hash,
    'n_units_train':         int(len(uid_arr)),
    'effective_pp_params':   PP_FIXED,
    'study_meta':            study_meta,
    'postprocess': {
        'best_agg':            pp_res['best_agg'],
        'pos_weights':         pp_res['pos_weights'].tolist() if pp_res['pos_weights'] is not None else None,
        'best_zero_clip':      float(pp_res['best_zero_clip']),
        'zero_clip_log_space': pp_res['zero_clip_log_space'],
        'position_method':     pp_res['position_method'],
        'agg_rmses':           {k: float(v) for k, v in pp_res['agg_rmses'].items()},
        'train_rmse':          float(pp_res['train_rmse']),
    },
}
with open(os.path.join(OUT_DIR, 'best_params.json'), 'w', encoding='utf-8') as f:
    json.dump(best_meta, f, indent=2, ensure_ascii=False, default=str)

# 3-5) die-level 예측 CSV
def _build_die_df(uid, die_id, position, pi, mu, pred, y_unit):
    df = pd.DataFrame({
        KEY_COL: uid, DIE_KEY_COL: die_id, 'position': position,
        'pi': pi, 'one_minus_pi': 1.0 - pi, 'mu': mu, 'pred': pred,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)
    return df

_build_die_df(
    uid_train_die, xs_train[DIE_KEY_COL].values, xs_train['position'].values,
    oof_die_pi, oof_die_mu, oof_die_pred, y_train_unit_s,
).to_csv(os.path.join(OUT_DIR, 'oof_die.csv'), index=False)
_build_die_df(
    uid_val_die, xs_val[DIE_KEY_COL].values, xs_val['position'].values,
    val_die_pi, val_die_mu, val_die_pred, y_val_unit_s,
).to_csv(os.path.join(OUT_DIR, 'val_die.csv'), index=False)
_build_die_df(
    uid_test_die, xs_test[DIE_KEY_COL].values, xs_test['position'].values,
    test_die_pi, test_die_mu, test_die_pred, y_test_unit_s,
).to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)

# 6-8) unit-level 예측 CSV (후처리 적용본 + health)
def _build_unit_df(unit_pred_df, y_unit):
    out = unit_pred_df.copy()
    out[TARGET_COL] = out[KEY_COL].map(y_unit)
    return out

_build_unit_df(pp_res['final_train_unit'], y_train_unit_s).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(pp_res['final_val_unit'],   y_val_unit_s  ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(pp_res['final_test_unit'],  y_test_unit_s ).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

# 9) optuna_*.db는 study.optimize가 학습 중 자동 저장 (별도 코드 불필요)

# 저장 파일 목록
print(f'\n저장 완료: {OUT_DIR}')
for fn in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, fn)) / 1024
    print(f'  {fn:30s}  {sz:10,.1f} KB')

# Colab이면 산출물 폴더를 zip으로 묶어 다운로드 (DB 포함)
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', f'zit_et_hpo_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'\n[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip))
except ImportError:
    pass

TypeError: float() argument must be a string or a real number, not 'NoneType'